# Overall movement-bout characterisation

Inspect speed, duration and travelled distance before deciding whether to filter
the premotor analyses. This notebook reads the **saved behavioural tables only**;
it does not rerun detection or load neural recordings.

The default selection follows the movement-bout heatmaps: use one headstage per
animal/session, preferring headstage 0 and retaining headstage-1-only sessions.
This avoids counting the same recorded behaviour twice. Headstage extractions can
differ slightly, so this is selection of a reference stream, not exact row deduplication.
Set `ONE_HEADSTAGE_PER_RECORDING=False` to inspect all saved session entries.
Original bout and session indices are preserved.

- Tracking is **red**; playback is **black**. Condition is assigned at bout onset.
- Duration ends at the start of the next qualifying immobility period. A bout can
  therefore include reversals and pauses shorter than the saved immobility threshold.
- `total_movement` is path length, including reversals. Absolute endpoint
  displacement is a separate quantity. Their ratio measures endpoint directness;
  it is not a count of reversals or a measure of trajectory smoothness.
- `mean_speed` is the saved mean of scaled `Speed_x`, not peak speed or initial speed.
  It describes the entire subsequent bout. Labels use position units, not assumed cm.
- Pooling bouts gives more weight to recordings with more bouts. A separate figure
  shows one median per recording and condition, grouped by animal.

All observed finite values enter the appropriate plots; no movement threshold is
applied here. Missing endpoint-dependent measurements are counted. Filter previews
report hypothetical exclusions and do not modify any saved bout or neural table.

In [ ]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ONE_HEADSTAGE_PER_RECORDING = True
SESSION_INDICES = None  # Optional original idx values, selected before headstage preference.
SAVE_FIGURES = True

data_dir = next(
    (parent / "data" for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "data/derived/movement_bouts/movement_bouts.pkl").is_file()),
    None,
)
if data_dir is None:
    raise FileNotFoundError("Cannot locate saved movement-bout tables.")

bout_dir = data_dir / "derived/movement_bouts"
movement_bouts = pd.read_pickle(bout_dir / "movement_bouts.pkl")
session_info = pd.read_pickle(bout_dir / "session_info.pkl")
parameters = json.loads((bout_dir / "parameters.json").read_text())
if not movement_bouts.index.is_unique or not session_info.index.is_unique:
    raise ValueError("Bout and session indices must be unique.")
if not movement_bouts["idx"].isin(session_info.index).all():
    raise ValueError("Some bouts refer to missing session_info entries.")

selected_sessions = session_info.sort_values(["animal", "session", "headstage"])
if SESSION_INDICES is not None:
    missing = set(SESSION_INDICES) - set(session_info.index)
    if missing:
        raise ValueError(f"Unknown session idx values: {sorted(missing)}")
    selected_sessions = selected_sessions.loc[selected_sessions.index.isin(SESSION_INDICES)]
if ONE_HEADSTAGE_PER_RECORDING:
    selected_sessions = selected_sessions.drop_duplicates(["animal", "session"], keep="first")
if selected_sessions.empty:
    raise ValueError("No sessions selected.")

selected_bouts = movement_bouts.loc[
    movement_bouts["idx"].isin(selected_sessions.index)
].copy()
analysis_bouts = selected_bouts.loc[selected_bouts["condition"].isin([0, 1])].copy()
analysis_bouts["duration_s"] = analysis_bouts["end_time_s"] - analysis_bouts["onset_time_s"]
analysis_bouts["net_displacement"] = (
    analysis_bouts["final_position"] - analysis_bouts["initial_position"]
)
analysis_bouts["absolute_displacement"] = analysis_bouts["net_displacement"].abs()
analysis_bouts["directness"] = analysis_bouts["absolute_displacement"].div(
    analysis_bouts["total_movement"].where(analysis_bouts["total_movement"].gt(0))
)
analysis_bouts["condition_name"] = analysis_bouts["condition"].map({0: "Tracking", 1: "Playback"})

CONDITIONS = [(0, "Tracking", "red"), (1, "Playback", "black")]
METRICS = [
    ("mean_speed", "Mean speed", "Position units/s"),
    ("duration_s", "Bout duration", "Seconds"),
    ("total_movement", "Travelled distance", "Position units"),
]
EXTRA_METRICS = [
    ("absolute_displacement", "Absolute endpoint displacement", "Position units"),
    ("directness", "Endpoint directness", "Displacement / distance"),
]

for metric, _, _ in METRICS + EXTRA_METRICS:
    values = analysis_bouts[metric].dropna().to_numpy(dtype=float)
    if not np.isfinite(values).all() or (values < -1e-10).any():
        raise ValueError(f"Nonfinite or negative values in {metric}.")
if analysis_bouts["duration_s"].dropna().le(0).any():
    raise ValueError("Completed bouts must have positive duration.")
if analysis_bouts["directness"].dropna().gt(1 + 1e-8).any():
    raise ValueError("Endpoint displacement exceeds travelled distance.")

scope = "one_headstage" if ONE_HEADSTAGE_PER_RECORDING else "all_entries"
selection_digest = hashlib.sha256(
    json.dumps(selected_sessions.index.tolist()).encode()
).hexdigest()[:8]
output_dir = bout_dir / "characterisation" / f"{scope}_{selection_digest}"

def save_figure(fig, name):
    if SAVE_FIGURES:
        output_dir.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_dir / f"{name}.png", dpi=160, bbox_inches="tight", facecolor="white")

plt.rcParams.update({
    "font.size": 10, "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "figure.facecolor": "white",
})
print(f"{len(analysis_bouts):,} bouts from {len(selected_sessions)} session entries, "
      f"{selected_sessions['animal'].nunique()} animals.")
print(f"Other onset conditions excluded: {len(selected_bouts) - len(analysis_bouts)}.")
print(f"Bout endpoint: next immobility lasting at least {parameters['MIN_IMMOBILITY_S']:g} s.")
print(f"Saved Speed_x multiplier: {parameters['SPEED_X_FACTOR']:g}.")
print(f"Figure directory: {output_dir}")

## Coverage and numerical summaries

Each row of `analysis_bouts` remains one original bout. Completeness is assessed
per metric, so an unfinished bout is not silently removed from every analysis.
The 5th, 50th, 95th and 99th percentiles describe the centre and tails.

In [ ]:
coverage_rows = []
summary_rows = []
for condition, name, _ in CONDITIONS:
    group = analysis_bouts.loc[analysis_bouts["condition"].eq(condition)]
    coverage_rows.append({
        "condition": name, "n_bouts": len(group), "n_sessions": group["idx"].nunique(),
        "unfinished_bouts": int(group["end_time_s"].isna().sum()),
        **{f"missing_{metric}": int(group[metric].isna().sum()) for metric, _, _ in METRICS},
    })
    for metric, label, units in METRICS + EXTRA_METRICS:
        values = group[metric].dropna()
        summary_rows.append({
            "condition": name, "metric": metric, "units": units, "n": len(values),
            "mean": values.mean(), "p05": values.quantile(.05),
            "median": values.median(), "p95": values.quantile(.95),
            "p99": values.quantile(.99), "max": values.max(),
        })
coverage = pd.DataFrame(coverage_rows).set_index("condition")
summary = pd.DataFrame(summary_rows).set_index(["metric", "condition"])
display(coverage)
display(summary.round(3))

## Overall distributions

Top: percentage of bouts in shared logarithmic bins, normalized separately for
tracking and playback. Bottom: cumulative percentage, with dashed median markers.
Logarithmic x axes retain the full positive range, including the longest bouts.
Zero values, if any, are reported explicitly and remain in the cumulative curves.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8.2), constrained_layout=True)
for column, (metric, label, units) in enumerate(METRICS):
    pooled = analysis_bouts[metric].dropna().to_numpy(dtype=float)
    positive = pooled[pooled > 0]
    if not len(positive):
        raise ValueError(f"No positive values for {metric}.")
    lower, upper = positive.min(), positive.max()
    if lower == upper:
        lower, upper = lower / 2, upper * 2
    edges = np.geomspace(lower * .999, upper * 1.001, 66)
    for condition, name, color in CONDITIONS:
        values = analysis_bouts.loc[analysis_bouts["condition"].eq(condition), metric].dropna().to_numpy(dtype=float)
        if not len(values):
            continue
        zeros = int((values == 0).sum())
        legend = f"{name} (n={len(values):,})" + (f"; {zeros} zeros" if zeros else "")
        axes[0, column].hist(
            values[values > 0], bins=edges,
            weights=np.full((values > 0).sum(), 100 / len(values)),
            histtype="step", color=color, linewidth=1.8, label=legend,
        )
        ordered = np.sort(values)
        axes[1, column].step(
            ordered, 100 * np.arange(1, len(ordered) + 1) / len(ordered),
            where="post", color=color, linewidth=1.8, label=name,
        )
        axes[1, column].axvline(np.median(values), color=color, linestyle="--", linewidth=1, alpha=.7)
    for row in (0, 1):
        axes[row, column].set_xscale("symlog", linthresh=lower / 2)
        axes[row, column].set_xlim(0 if (pooled == 0).any() else lower * .85, upper * 1.15)
        axes[row, column].set_xlabel(f"{label} ({units.lower()})")
        axes[row, column].grid(axis="y", alpha=.16)
    axes[0, column].set_title(label)
    axes[0, column].set_ylabel("Bouts per bin (%)")
    axes[1, column].set_ylabel("Cumulative bouts (%)")
    axes[1, column].set_ylim(0, 100)
    axes[0, column].legend(frameon=False, fontsize=8)
fig.suptitle(f"Movement-bout distributions | {len(analysis_bouts):,} bouts, {len(selected_sessions)} session entries", fontsize=14)
save_figure(fig, "01_overall_distributions")
plt.show()

## Path length, duration and endpoint displacement

Long bouts can cover much more than the 0–28 position range because path length
adds repeated movement. In the bottom panels, the dashed diagonal marks distance
equal to endpoint displacement. Points far below it finish close to their start
relative to how much they travelled. These are not necessarily small movements.
The displacement axis is linear near zero and logarithmic above 0.5 position units.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex="row", sharey="row", constrained_layout=True)
for column, (condition, name, color) in enumerate(CONDITIONS):
    group = analysis_bouts.loc[analysis_bouts["condition"].eq(condition)]
    valid = group[["duration_s", "total_movement"]].dropna()
    valid = valid.loc[valid.gt(0).all(axis=1)]
    axes[0, column].scatter(valid["duration_s"], valid["total_movement"], s=7, alpha=.18, color=color, edgecolors="none", rasterized=True)
    axes[0, column].set_xscale("log")
    axes[0, column].set_yscale("log")
    axes[0, column].set_xlabel("Bout duration (s)")
    axes[0, column].set_ylabel("Travelled distance (position units)")
    axes[0, column].set_title(f"{name}: duration and distance")

    valid = group[["total_movement", "absolute_displacement"]].dropna()
    valid = valid.loc[valid["total_movement"].gt(0)]
    axes[1, column].scatter(valid["total_movement"], valid["absolute_displacement"], s=7, alpha=.18, color=color, edgecolors="none", rasterized=True)
    identity = np.geomspace(valid["total_movement"].min(), 28, 100)
    axes[1, column].plot(identity, identity, linestyle="--", color="0.45", linewidth=1, label="Displacement = distance")
    axes[1, column].set_xscale("log")
    axes[1, column].set_yscale("symlog", linthresh=.5)
    axes[1, column].set_ylim(0, 30)
    axes[1, column].set_xlabel("Travelled distance (position units)")
    axes[1, column].set_ylabel("Absolute endpoint displacement (position units)")
    axes[1, column].set_title(f"{name}: endpoint versus travelled distance")
    axes[1, column].legend(frameon=False, fontsize=8, loc="upper left")
    for ax in axes[:, column]:
        ax.grid(alpha=.13)
fig.suptitle("Which bouts are long, and which finish near their starting point?", fontsize=14)
save_figure(fig, "02_distance_duration_displacement")
plt.show()

## Recording medians by animal

One dot is a session entry's median in one condition. Lines connect tracking and
playback from the same entry; horizontal offsets separate entries within an animal.
This view gives each selected recording one point instead of weighting it by bout count.

In [ ]:
session_medians = analysis_bouts.groupby(["idx", "condition"])[[m[0] for m in METRICS]].median()
animals = selected_sessions["animal"].drop_duplicates().tolist()
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), constrained_layout=True)
for animal_number, animal in enumerate(animals):
    indices = selected_sessions.index[selected_sessions["animal"].eq(animal)]
    offsets = np.linspace(-.27, .27, len(indices)) if len(indices) > 1 else [0]
    for idx, offset in zip(indices, offsets):
        x = animal_number + offset
        for ax, (metric, label, units) in zip(axes, METRICS):
            pair = session_medians.loc[idx, metric].reindex([0, 1])
            if pair.notna().all():
                ax.plot([x - .025, x + .025], pair.to_numpy(), color=".75", linewidth=.8, zorder=1)
            for condition, name, color in CONDITIONS:
                if pd.notna(pair.loc[condition]):
                    ax.scatter(x + (-.025 if condition == 0 else .025), pair.loc[condition], color=color, s=23, alpha=.8, zorder=2)
for ax, (_, label, units) in zip(axes, METRICS):
    ax.set_xticks(range(len(animals)), animals, rotation=15)
    ax.set_title(label)
    ax.set_ylabel(f"Median ({units.lower()})")
    ax.set_ylim(bottom=0)
    ax.grid(axis="y", alpha=.16)
axes[0].plot([], [], "o", color="red", label="Tracking")
axes[0].plot([], [], "o", color="black", label="Playback")
axes[0].legend(frameon=False)
fig.suptitle("Recording medians: each selected session contributes one paired comparison", fontsize=13)
save_figure(fig, "03_recording_medians_by_animal")
plt.show()

## Hypothetical filters: how many bouts would be removed?

Each curve/table row considers **one cutoff at a time**, without applying it.
The denominator is the number of bouts with a finite value for that metric;
missing values are reported separately. Minimum cutoffs exclude values strictly
below the cutoff; maximum cutoffs exclude values strictly above it.

These are example thresholds to inspect, not recommended defaults. In particular,
filtering on endpoint displacement selects against return movements, while a
duration cutoff changes the mix of movement episodes. Counts should be checked
by animal/session before selecting a rule for the neural analyses.

In [ ]:
filter_specs = [
    ("mean_speed", "min", [1, 2], "Minimum mean speed (position units/s)", "Minimum speed"),
    ("duration_s", "max", [5, 10, 20], "Maximum bout duration (s)", "Maximum duration"),
    ("total_movement", "min", [1, 2, 5], "Minimum travelled distance (position units)", "Minimum travelled distance"),
    ("directness", "min", [.1, .25, .5], "Minimum endpoint directness", "Minimum directness"),
]
fig, axes = plt.subplots(2, 2, figsize=(13, 8.8), constrained_layout=True)
filter_rows = []
for ax, (metric, rule, cutoffs, xlabel, title) in zip(axes.flat, filter_specs):
    finite = analysis_bouts[metric].dropna()
    if metric == "duration_s":
        grid = np.geomspace(finite.min(), finite.max(), 220)
        ax.set_xscale("log")
    elif metric == "total_movement":
        positive = finite.loc[finite.gt(0)]
        grid = np.r_[0, np.geomspace(positive.min(), positive.quantile(.99), 219)]
        ax.set_xscale("symlog", linthresh=1)
    elif metric == "directness":
        grid = np.linspace(0, 1, 220)
    else:
        grid = np.linspace(0, finite.quantile(.99), 220)
    for condition, name, color in CONDITIONS:
        group = analysis_bouts.loc[analysis_bouts["condition"].eq(condition)]
        values = np.sort(group[metric].dropna().to_numpy(dtype=float))
        if not len(values):
            continue
        excluded = np.searchsorted(values, grid, side="left") if rule == "min" else len(values) - np.searchsorted(values, grid, side="right")
        ax.plot(grid, 100 * excluded / len(values), color=color, linewidth=1.8, label=name)
        for cutoff in cutoffs:
            n_excluded = int((values < cutoff).sum() if rule == "min" else (values > cutoff).sum())
            ax.scatter(cutoff, 100 * n_excluded / len(values), color=color, s=22)
            filter_rows.append({
                "metric": metric, "rule": rule, "cutoff": cutoff, "condition": name,
                "n_finite": len(values), "n_missing": len(group) - len(values),
                "n_excluded": n_excluded, "excluded_percent": 100 * n_excluded / len(values),
            })
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Finite bouts excluded (%)")
    ax.set_ylim(0, 100)
    ax.grid(alpha=.16)
    ax.legend(frameon=False)
fig.suptitle("Filter sensitivity: independent cutoffs, no filtering applied", fontsize=14)
save_figure(fig, "04_filter_sensitivity")
plt.show()

# Also quantify endpoint-displacement thresholds for comparison with path length.
for cutoff in [1, 2, 5]:
    for condition, name, _ in CONDITIONS:
        group = analysis_bouts.loc[analysis_bouts["condition"].eq(condition)]
        values = group["absolute_displacement"].dropna()
        n_excluded = int(values.lt(cutoff).sum())
        filter_rows.append({
            "metric": "absolute_displacement", "rule": "min", "cutoff": cutoff,
            "condition": name, "n_finite": len(values), "n_missing": len(group) - len(values),
            "n_excluded": n_excluded,
            "excluded_percent": 100 * n_excluded / len(values) if len(values) else np.nan,
        })
filter_sensitivity = pd.DataFrame(filter_rows)
display(filter_sensitivity.round(2))

## Inspect the extremes before choosing a cutoff

These are the five largest bouts for each main metric, with original bout/session
IDs and onset times so they can be located in the existing session browser in
`analyse_movement_bouts.ipynb`. Rows can appear in multiple lists. A high value
does not by itself identify an artefact. Long bouts also warrant checking whether
they cross condition/block transitions; the saved bout table does not record that.

In [ ]:
tail_tables = []
for metric, _, _ in METRICS:
    tail = analysis_bouts.nlargest(5, metric)[[
        "idx", "animal", "headstage", "session", "condition_name", "onset_time_s",
        "duration_s", "mean_speed", "total_movement", "absolute_displacement", "directness",
    ]].copy()
    tail.index.name = "bout_id"
    tail_tables.append(tail.assign(ranked_by=metric).reset_index())
extreme_bouts = pd.concat(tail_tables, ignore_index=True).set_index(["ranked_by", "bout_id"])
display(extreme_bouts.round(3))

# A numerical cross-check of the saved speed definition, not a replacement metric.
complete = analysis_bouts.dropna(subset=["duration_s", "total_movement", "mean_speed"])
path_speed = complete["total_movement"] / complete["duration_s"]
relative_error = (path_speed - complete["mean_speed"]).abs().div(complete["mean_speed"].where(complete["mean_speed"].gt(0)))
print(f"Median relative difference between saved mean speed and distance/duration: {relative_error.median():.3g}.")
print("These measures are closely linked; speed, duration and distance are not three independent filters.")

if SAVE_FIGURES:
    source_sha256 = {
        name: hashlib.sha256((bout_dir / name).read_bytes()).hexdigest()
        for name in ["movement_bouts.pkl", "session_info.pkl", "parameters.json"]
    }
    tables = {
        "coverage": coverage, "summary": summary,
        "session_medians": session_medians,
        "filter_sensitivity": filter_sensitivity, "extreme_bouts": extreme_bouts,
        "selected_sessions": selected_sessions,
        "selected_bout_ids": analysis_bouts.index,
        "source_sha256": source_sha256,
        "settings": {
            "one_headstage_per_recording": ONE_HEADSTAGE_PER_RECORDING,
            "requested_session_indices": SESSION_INDICES,
            "selected_session_indices": selected_sessions.index.tolist(),
            "condition_at_onset": True, "movement_filters_applied": False,
            "bout_detection_parameters": parameters,
        },
    }
    pd.to_pickle(tables, output_dir / "characterisation_tables.pkl")
    (output_dir / "provenance.json").write_text(
        json.dumps({"source_sha256": source_sha256, "settings": tables["settings"]}, indent=2) + "\n",
        encoding="utf-8",
    )
    print(f"Saved four PNG figures, summary tables and source provenance to {output_dir}.")